# 논문2 XGBoost 산불 피해 규모 분류 모델 정리

논문2: **기계학습을 활용한 산불 피해 규모 예측: 기상 및 환경 변수를 중심으로**

이 노트북은 논문2 모델을 강원도 날씨 데이터에 적용하기 위해 정리한 파일입니다.

핵심 결론:

- 논문2는 산불 피해 규모를 **소형 / 중형 / 대형**으로 분류합니다.
- 비교 모델은 `Random Forest`, `XGBoost`, `SVM`입니다.
- 논문 결과에서 가장 좋은 모델은 **XGBoost**입니다.
- 우리 강원도 날씨 데이터는 논문2 입력 피처 형태로 변환했습니다.
- 하지만 현재 프로젝트에는 소형/중형/대형 타깃 라벨이 완전하지 않아 실제 학습/예측은 아직 불가능합니다.

## 1. 경로 정리

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('../..').resolve()
paper2_input_path = ROOT / 'data/modeling/paper2_xgboost_gangwon_weather_input.csv'
paper2_script_path = ROOT / 'jgy/modeling/prepare_paper2_xgboost_input.py'

print('논문2 입력 생성 스크립트:', paper2_script_path)
print('존재 여부:', paper2_script_path.exists())
print()
print('논문2 강원도 입력 데이터:', paper2_input_path)
print('존재 여부:', paper2_input_path.exists())

## 2. 논문2 모델 설명

논문2는 산불 발생 이후 피해 규모를 빠르게 분류하는 모델을 만듭니다.

분류 기준:

```text
대형 산불: 피해면적 30ha 이상 또는 지속시간 24시간 초과
중형 산불: 피해면적 5ha 이상 30ha 미만 또는 지속시간 8시간 이상 24시간 이하
소형 산불: 나머지
```

사용 모델:

```text
Random Forest
XGBoost
SVM
```

논문 결과:

```text
Random Forest 정확도: 0.95, 대형 재현율: 0.79
XGBoost 정확도: 0.96, 대형 재현율: 0.89
SVM 정확도: 0.90, 대형 재현율: 0.58
```

따라서 실무적으로는 대형 산불을 더 잘 잡는 `XGBoost`가 가장 적합하다고 해석합니다.

## 3. 논문2 입력 변수 구조

논문2는 산불 지속시간 동안의 시간 단위 기상자료를 요약해서 사용합니다.

주요 변수:

```text
기온
풍속
습도
기압 차
SPI1, SPI2, SPI3
침엽수 비율
계절
시도/지역
```

각 기상 변수는 평균, 최대-최소 차이, 변화량 평균, 절댓값 변화량 평균, 변화량 비율 같은 파생 변수로 변환합니다.

우리 데이터에는 정식 SPI와 침엽수 비율이 아직 없어서, 현재는 다음처럼 처리했습니다.

```text
SPI1/2/3: 월 강수량 기반 proxy 변수로 대체
침엽수 비율: 아직 없음
시도: 강원도 단일 지역이라 별도 지역 변수 대신 기후권역/기후지형유형 사용
```

## 4. 강원도 날씨 데이터 변환 결과

In [ ]:
paper2_input = pd.read_csv(paper2_input_path, encoding='utf-8-sig')

print('행/열:', paper2_input.shape)
print('날짜 범위:', paper2_input['date'].min(), '~', paper2_input['date'].max())
paper2_input.head()

## 5. 생성된 주요 피처 확인

In [ ]:
paper2_input.columns.tolist()

In [ ]:
summary_cols = [
    'temp_mean_C', 'temp_range_C',
    'wind_mean_m_s', 'wind_range_m_s',
    'humidity_mean_pct', 'humidity_range_pct',
    'pressure_diff_mean_hPa', 'pressure_diff_range_hPa',
    'spi1_proxy', 'spi2_proxy', 'spi3_proxy'
]

paper2_input[summary_cols].describe().round(3)

## 6. 현재 결과와 한계

현재 생성된 파일:

```text
data/modeling/paper2_xgboost_gangwon_weather_input.csv
```

결과:

```text
행 수: 67,252
컬럼 수: 32
날짜 범위: 2020-01-01 ~ 2021-12-31
```

하지만 이 파일은 아직 `소형/중형/대형` 정답 라벨이 없는 입력 피처 파일입니다.

실제 논문2 방식의 XGBoost 학습을 하려면 아래가 더 필요합니다.

```text
1. 산불별 피해면적(ha)
2. 산불별 지속시간
3. 소형/중형/대형 라벨
4. 가능하면 정식 SPI1/SPI2/SPI3
5. 침엽수 비율 또는 토지피복/산림 구조 변수
```

즉, 현재는 **논문2 모델 입력 데이터 생성**까지 완료된 상태입니다.

실제 XGBoost 예측 결과는 아직 없습니다.

## 7. 임시 예측 결과

진짜 XGBoost 학습 모델이 아니라, 논문2에서 중요하다고 한 기상 조건을 조합한 임시 예측입니다.

생성 파일:

`	ext
data/modeling/paper2_temporary_gangwon_size_predictions.csv
`

결과:

`	ext
소형: 47,061
중형: 16,821
대형: 3,370
`

임시 기준:

`	ext
상위 5%: 대형
상위 30% 중 대형 제외: 중형
나머지: 소형
`

최고 점수 사례:

`	ext
기상셀ID: YS_0032
날짜: 2021-03-16
기후권역: 영서
기후지형유형: 고지·산간형
계절: 봄
임시 피해규모 점수: 0.8558
임시 예측 라벨: 대형
`

주의: 이 결과는 실제 산불 피해규모 예측 모델 결과가 아니라, 현재 가진 날씨 데이터만으로 만든 참고용 임시 분류입니다.


In [ ]:
temporary_path = ROOT / 'data/modeling/paper2_temporary_gangwon_size_predictions.csv'
temporary = pd.read_csv(temporary_path, encoding='utf-8-sig')

print('행/열:', temporary.shape)
print(temporary['paper2_temporary_size_pred'].value_counts())

temporary.sort_values('paper2_temporary_damage_score_0_1', ascending=False).head(10)


## 7. 다음 단계

```text
완료:
- 논문2 모델 구조 파악
- 강원도 통합 시간단위 날씨를 논문2 피처 구조로 변환
- XGBoost 입력용 CSV 생성

아직 필요:
- 산불 피해면적/지속시간이 있는 학습 데이터 확보
- 소형/중형/대형 라벨 생성
- XGBoost 학습 코드 작성
- 변수 중요도와 예측 결과 확인
```